# Online Retail II Exploration

This notebook loads both yearly sheets from the Online Retail II workbook into pandas, validates and prepares the data, flags questionable rows, and saves reviewable outputs.

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Configure File Paths

In [ ]:
PROJECT_ROOT = Path.cwd()
INPUT_PATH = PROJECT_ROOT / "data" / "online_retail_II.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "reports"

INPUT_PATH, OUTPUT_DIR

## 3. Read Excel Workbook into a DataFrame

In [ ]:
sheets = pd.read_excel(INPUT_PATH, sheet_name=None)
df = pd.concat(
    [sheet.assign(source_sheet=sheet_name) for sheet_name, sheet in sheets.items()],
    ignore_index=True,
)

print(f"Loaded {len(sheets)} sheets and {len(df):,} rows.")
df.head()

## 4. Inspect the DataFrame

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.info()
df.head(10)

## 5. Validate Expected Columns

In [ ]:
expected_columns = {
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
    "source_sheet",
}
missing_columns = expected_columns.difference(df.columns)
extra_columns = set(df.columns).difference(expected_columns)

assert not missing_columns, f"Missing columns: {sorted(missing_columns)}"
print("Schema is valid.")
print("Extra columns:", sorted(extra_columns) if extra_columns else "None")

## 6. Convert Data Types

In [ ]:
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Revenue"] = df["Quantity"] * df["Price"]

df[["Quantity", "Price", "InvoiceDate", "Revenue"]].dtypes

## 7. Identify Invalid Rows

In [ ]:
df["is_cancelled"] = df["Invoice"].astype("string").str.upper().str.startswith("C", na=False)

missing_invoice = df["Invoice"].isna()
missing_stock_code = df["StockCode"].isna()
missing_price = df["Price"].isna()
non_numeric_quantity = df["Quantity"].isna()
invalid_date = df["InvoiceDate"].isna()

invalid_mask = (
    missing_invoice
    | missing_stock_code
    | missing_price
    | non_numeric_quantity
    | invalid_date
)
invalid_rows = df.loc[invalid_mask].copy()
invalid_rows["quality_flag"] = "invalid_core_data"
invalid_rows.loc[df.loc[invalid_mask, "Customer ID"].isna(), "quality_flag"] = "missing_customer_id"

print(f"Flagged rows: {len(invalid_rows):,}")
invalid_rows.head()

## 8. Review the First Rows and Missing Values

In [ ]:
display(df.head(10))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))
display(df["Country"].value_counts(dropna=False).head(20).to_frame("row_count"))

## 9. Aggregate Sales Metrics

In [ ]:
sales_rows = df.loc[~df["is_cancelled"] & df["Revenue"].notna()].copy()
sales_rows["month"] = sales_rows["InvoiceDate"].dt.to_period("M").astype("string")

monthly_revenue = sales_rows.groupby("month", as_index=False)["Revenue"].sum().sort_values("month")
top_products = (
    sales_rows.groupby("Description", dropna=False)["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .rename("Revenue")
    .reset_index()
)
top_countries = (
    sales_rows.groupby("Country", dropna=False)["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .rename("Revenue")
    .reset_index()

metrics = pd.Series(
    {
        "revenue": sales_rows["Revenue"].sum(),
        "orders": sales_rows["Invoice"].nunique(),
        "customers": sales_rows["Customer ID"].nunique(),
        "cancelled_rows": int(df["is_cancelled"].sum()),
        "flagged_rows": len(invalid_rows),
    },
    name="value",
)
display(metrics.to_frame())
display(monthly_revenue.head())
display(top_products)
display(top_countries)

## 10. Save Processed Outputs

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

invalid_rows.to_csv(OUTPUT_DIR / "notebook_invalid_rows.csv", index=False)
monthly_revenue.to_csv(OUTPUT_DIR / "notebook_monthly_revenue.csv", index=False)
top_products.to_csv(OUTPUT_DIR / "notebook_top_products.csv", index=False)
top_countries.to_csv(OUTPUT_DIR / "notebook_top_countries.csv", index=False)
sales_rows.to_csv(OUTPUT_DIR / "notebook_sales_rows.csv", index=False)

print(f"Saved notebook outputs to {OUTPUT_DIR.resolve()}")